# Notebook 1: Build a graph "backbone" from structured data sources

Import the Python library dependencies.

In [ ]:
import json
import pathlib

from sz_semantics import SzClient, Thesaurus
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions

## Open Data

In this tutorial we will construct and analyze an _investigative graph_, connecting "risk" data and "link" data within a graph. From this we can show patterns of criminal _tradecraft_ used in money laundering, tax evasion, and so on.

We have selected "slices" of open data from two providers, where elements connect to produce interesting subgraphs about fraud networks:

  - <https://www.opensanctions.org/>
  - <https://www.openownership.org/>

### OpenSanctions

[OpenSanctions](https://www.opensanctions.org/) provides the "risk" category of data.
In other words, this describes people and organizations who are known risks for FinCrime.
There is also the [`yente`](https://github.com/opensanctions/yente) API which provides
HTTP endpoints based on the [_FollowTheMoney_](https://followthemoney.tech/) data model
used for investigations and [OSInt](https://osintframework.com/).

### Open Ownership

[Open Ownership](https://www.openownership.org/) provides the "link" category of data.
This describes [_ultimate beneficial ownership_](https://www.beneficialownership.co.uk/)
(UBO) details: "Who owns how much of what, and who actually has controlling interest?"
There's also the [_Beneficial Ownership Data Standard_](https://standard.openownership.org/en/0.4.0/)
(BODS) which is an open standard providing guidance for collecting, sharing, and using
high-quality beneficial ownership data, to support corporate ownership transparency.

Open Ownership has partnered with [GLEIF](https://www.gleif.org/) to
launch the [_Global Open Data Integration Network_](https://godin.gleif.org)
(GODIN) to promote open standards across the world for data interoperability
among these kinds of datasets related to investigating transnational corruption.

Note: there is also a repository which publishes these datasets, already formatted as JSONL files for use in Senzing <https://www.opensanctions.org/docs/bulk/senzing/>

However, those full sources are a lot to download, so for this tutorial we're using selected "slices" which will produce interesting subgraphs.
To download these slices of the `OpenSanctions` and `Open Ownership` datasets:

In [ ]:
!wget https://raw.githubusercontent.com/DerwenAI/senzing_starter_kit/refs/heads/main/senzing_rootfs/data/open-sanctions.json \
  -O data/open-sanctions.json

In [ ]:
!wget https://raw.githubusercontent.com/DerwenAI/senzing_starter_kit/refs/heads/main/senzing_rootfs/data/open-ownership.json \
  -O data/open-ownership.json

This should create the two JSONL files in the `data` subdirectory:
* `data/open-sanctions.json`
* `data/open-ownership.json`

In [ ]:
!ls -lta data/*.json

Let's examine the results, by JSON _pretty-printing_ the first line in each file.

In [ ]:
!head -1 data/open-sanctions.json | python3 -m json.tool

Note the _risk_ elements in this data, which are `"TOPIC"` files within the `"RISKS"` category.

Then take a look at the _link_ elements:

In [ ]:
!head -1 data/open-ownership.json | python3 -m json.tool

## Run entity resolution

Make sure you have already launched the [`serve-grpc` container](https://hub.docker.com/r/senzing/serve-grpc) and have it running in the background, by executing the following command line in another terminal window:

```bash
docker run -it --publish 8261:8261 --rm senzing/serve-grpc
```

In [ ]:
!docker ps

Next we'll use [`sz_semantics`](https://github.com/senzing-garage/sz-semantics/) library to call the Senzing SDK via the [gRPC](https://grpc.io/) server running in that Docker container.

To get started on _entity resolution_, first we need to specify a namespace for the [data sources](https://senzing.zendesk.com/hc/en-us/articles/115002897308-Data-Source-Records-DSRs-Explained) to use in Senzing.

In [ ]:
data_sources: dict[ str, str ] = {
    "OPEN-SANCTIONS": "data/open-sanctions.json",
    "OPEN-OWNERSHIP": "data/open-ownership.json",
}

We need to configure how to reach the the Senzing SDK which is running as a [_microservice_](https://aws.amazon.com/microservices/).
See the [gRPC server](https://github.com/senzing-garage/serve-grpc) documentation for more details.

In [ ]:
config: dict[ str, dict ] = {
    "sz": { "grpc_server": "localhost:8261" }
}

Now we have the two parts needed to configure the Senzing SDK.

In [ ]:
sz: SzClient = SzClient(config, data_sources)

Then in one line of Python, run [entity resolution](https://senzing.com/what-is-entity-resolution/) on the named datasets.

In [ ]:
ents_batch: dict = sz.entity_resolution(data_sources)

Let's examine the JSON returned from a Senzing SDK ["GET_ENTITY"](https://senzing.com/docs/tutorials/get_entity_response/) call on the first resolved entity.

In [ ]:
one_get: str = sz.get_entity(1)
json.loads(one_get)

Next, we will represent each entity in RDF.

## Generate a domain-specific thesaurus

As a next step toward generating graph "building blocks" in RDF, initialize a `Thesaurus` instance and load the [Senzing taxonomy](https://github.com/senzing-garage/sz-semantics/blob/main/domain.ttl) into it.

In [ ]:
thesaurus: Thesaurus = Thesaurus()
thesaurus.load_source(Thesaurus.DOMAIN_TTL)

Let's examine how one JSON response from the Senzing SDK gets represented in RDF:

In [ ]:
for rdf_frag in thesaurus.parse_iter(one_get, language = "en"):
    print(rdf_frag)

Next we'll iterate through all the resolved entities:
1. generate RDF fragments from from the JSON response for "GET_ENTITY"
2. collect these RDF triples into a list
3. prepend the RDF namespace prefixes onto the load
4. load these triples into the RDF graph in the `Thesaurus` (internally using `RDFlib`)

In [ ]:
for ent_json in sz.sz_engine.export_json_entity_report_iterator():
    for rdf_frag in thesaurus.parse_iter(ent_json, language = "en"):
        thesaurus.load_source_text(
            Thesaurus.RDF_PREAMBLE + rdf_frag,
            format = "turtle",
        )

How many triples have we just loaded?

In [ ]:
len(thesaurus.rdf_graph)

Now the RDF graph includes triples for both the Senzing taxonomy and the generated domain-specific thesaurus, so serialize these results into the `thesaurus.ttl` file. This is written in ["Turtle"](https://medium.com/wallscope/understanding-linked-data-formats-rdf-xml-vs-turtle-vs-n-triples-eb931dbe9827) format, which is arguably much simpler to read, automatically verified, and also more _composable_.

In [ ]:
thesaurus_path: pathlib.Path = pathlib.Path("thesaurus.ttl")

thesaurus.save_source(
    thesaurus_path,
    format = "turtle",
)

Let's examine the triples in the top part of that file.

In [ ]:
!head -200 thesaurus.ttl

Now we have an RDF _semantic graph_ to use in the subsequent steps.
At this point, let's examine the [SKOS](https://www.w3.org/2004/02/skos/) taxonomy used by Senzing and how it integrates with other related vocabularies.
In another browser tab, open to <https://github.com/senzing-garage/sz-semantics/wiki/ns>

---